# Haykin — AR(2) NLMS prediction

統一 AR(2) convention：x[k]=−0.1x[k−1]+0.8x[k−2]+v[k]，目標係數 [−0.1,0.8]。unit-variance process 的 innovation variance 為 0.27。

來源：`Haykin-Chap7.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Parameters and generator

In [ ]:
n_samples, n_trials, seed = 300, 100, 30
true_weights = np.array([-0.1, 0.8])
noise_floor = data.ar2_innovation_variance()
make_data = partial(data.ar_prediction, n_samples, coefficients=true_weights, variance=1.0)
print("Innovation variance:", noise_floor)

## 3. Models and training

In [ ]:
models = {f"NLMS τ={tau}": partial(af.NLMS, filter_order=1, mu=0.2, tau=tau) for tau in (0.5, 0.25, 0.75)}
results = training.compare(models, make_data, n_trials=n_trials, seed=seed)

## 4. MSE, coefficient trajectories and estimation error

In [ ]:
fig, grid = plot.axes(rows=2)
plot.learning(results, ax=grid[0, 0], noise_floor=noise_floor, window=11)
plot.learning(results, quantity="msd", ax=grid[1, 0], title="AR coefficient MSD", window=11)
plot.show(fig)
for result in results.values():
  plot.coefficients(result, reference=true_weights, title=result.name)
  plot.show()
training.summary(results, tail=100, noise_floor=noise_floor)